# Encoding TEI with LLMs

This notebook demonstrates how to use Large Language Models (LLMs) to assist in the encoding of texts in the Text Encoding Initiative (TEI) format. The TEI is a standard for representing texts in digital form, and LLMs can help automate and enhance the encoding process.

Ce notebook montre comment utiliser des grands modèles de langue (LLMs) pour assister à l'encodage de textes au format Text Encoding Initiative (TEI). La TEI est un standard permettant de représenter des textes en forme numérique, et les LLMs peuvent automatiser et améliorer le processus d'encodage. 


In [15]:
pip install dotenv #pour lire les fichiers cachés

Note: you may need to restart the kernel to use updated packages.


In [16]:
pip install openai

Note: you may need to restart the kernel to use updated packages.


In [17]:
pip install google-genai

Note: you may need to restart the kernel to use updated packages.


In [18]:
pip install rapidfuzz

Note: you may need to restart the kernel to use updated packages.


In [19]:
import lxml
from lxml import etree

### Utiliser GoogleAI API

Obtenir une clé API gratuite de : https://aistudio.google.com/app/api-keys

Enregistrer la clé API dans un fichier `.env` comme `GOOGLE_API_KEY=your_api_key`

limite de 14k requêtes par jour:

* gemma-3-4b
* gemma-3-12b
* gemma-3-27b

limite de 20 requêtes par jour:
* gemini-2.5-flash
* gemini-3-flash



In [6]:
from pathlib import Path
import os
from openai import OpenAI
from google import genai
from tqdm import tqdm

from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
## **** A CONFIGURER ****
provider = "openrouter" #"googleai" or "googleai"

In [ ]:
def build_user_prompt(text: str) -> str:
    return f"""TEXTE À ENCODER :
{text}

TEXTE ENCODE EN XML TEI :
"""


def openrouter(client, model_name, pre_prompt, text):
    user_prompt = build_user_prompt(text)
    # Gemma → pas de system message
    if "gemma" in model_name.lower() or "mistral" in model_name.lower():

        full_prompt = pre_prompt + "\n\n" + user_prompt

        response = client.chat.completions.create(
            model=model_name,
            temperature=0,
            messages=[
                {"role": "user", "content": full_prompt},
            ],
        )
    # Autres modèles → messages normaux
    else:
        response = client.chat.completions.create(
            model=model_name,
            temperature=0,
            messages=[
                {"role": "system", "content": pre_prompt},
                {"role": "user", "content": user_prompt},
            ],
        )
    return response.choices[0].message.content.strip()


def googleai(client, model_name, pre_prompt, text):
    user_prompt = build_user_prompt(text)
    response = client.models.generate_content(
        model=model_name,
        contents=pre_prompt + "\n\n" + user_prompt
    )
    return response.text


def tei_encoding(pre_prompt, text, provider, client, model_name):
    try:
        if provider == "openrouter":
            corrected_text= openrouter(client, model_name, pre_prompt, text)
        elif provider == "googleai":
            corrected_text = googleai(client, model_name, pre_prompt, text)
        else:
            raise ValueError("Provider inconnu")
        
        # Éliminer ```xml au début et ``` à la fin s'ils existent
        if corrected_text.startswith("```xml"):
            corrected_text = corrected_text[len("```xml"):].strip()
        if corrected_text.endswith("```"):
            corrected_text = corrected_text[:-len("```")].strip()

        return corrected_text

    except Exception as e:
        print(f"Erreur {provider}: {e}")
        return text

In [22]:
if provider == "googleai":
    client = genai.Client(
        api_key=os.getenv("GOOGLE_API_KEY")
        )
elif provider == "openrouter":
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.getenv("OPENROUTER_API_KEY"),
    )

In [ ]:
#Fonction permettant de nettoyer les sorties de Gemma
def clean_xml_output(text: str) -> str:
    """Retire les balises markdown ```xml ... ``` que certains modèles (notamment Gemma) 
    ajoutent autour du XML généré."""
    text = text.strip()
    if text.startswith("```xml"):
        text = text[len("```xml"):].strip()
    elif text.startswith("```"):
        text = text[len("```"):].strip()
    if text.endswith("```"):
        text = text[:-len("```")].strip()
    return text

### A/ Programme permettant d'obtenir des fichiers xml à partir d'un chemin spécifique (par exemple, contenant les fichiers produits par un modèle, une sratégie et un niveau d'encodage)

In [ ]:

## **** A CONFIGURER ****
provider = "openrouter" #"googleai" or "googleai"

#models = ["openai/gpt-5-mini", "google/gemma-3-27b-it", "openai/gpt-5.4-mini", "google/gemini-3.1-flash-lite-preview"]
#models=["google/gemma-3-27b-it"]
#models=["openai/gpt-5-mini"]
#models=["openai/gpt-5.4-mini"]
models=["google/gemini-3.1-flash-lite-preview"]

#A compléter avec le chemin vers les fichiers qui me serviront de base
ocr_path_name = "output/niveau_parallele_2.1/gemini-3.1-flash-lite-preview/FS"
ocr_files = sorted(Path(ocr_path_name).rglob("*.xml"))

#Modifier le niveau des prompts à chaque fois
prompts = sorted(Path("prompts/niveau_parallele_3.1").rglob("*.txt"))

for prompt in prompts:
    with open(prompt) as f:
        pre_prompt = f.read()

    print("* "+prompt.stem)

    for model_name in models:
        if provider == "openrouter":
            model_short = model_name.split("/")[1].split(":")[0]
        elif provider == "googleai":
            model_short = model_name

        print("** "+ model_short)

        #Compléter avec le niveau pour créer un dossier si le dossier des sorties n'existe pas
        output_path = Path(f"./output/niveau_parallele_3.1/{model_short}")
        output_path.mkdir(parents=True, exist_ok=True)

        output_path = Path(f"./output/niveau_parallele_3.1/{model_short}/{prompt.stem}")
        output_path.mkdir(parents=True, exist_ok=True)

        for ocr_file in tqdm(ocr_files):
            ocr_text = ocr_file.read_text(encoding="utf-8", errors="replace")

            #Si le fichier existe déjà, passer
            output_file = output_path / f"{ocr_file.stem}.xml"
            if output_file.exists():
                continue

            corrected_text = tei_encoding(pre_prompt, ocr_text, provider, client, model_name)
            corrected_text = clean_xml_output(corrected_text)
            
            #Vérifier si le XML produit est valide (max 2 tentatives)
            for _ in range(2):
                try:
                    lxml.etree.fromstring(corrected_text.encode("utf-8"))
                    break
                except lxml.etree.XMLSyntaxError as e:
                    print(f"Retry: Erreur XML pour le fichier {ocr_file}: {e}")
                    corrected_text = tei_encoding(pre_prompt, ocr_text, provider, client, model_name)
            else:
                print(f"Impossible de corriger le fichier {ocr_file}")
                #continue

            #Écrire le texte valide dans le fichier de sortie
            output_file.write_text(corrected_text, encoding="utf-8", errors="replace")



* FS-R
** gemini-3.1-flash-lite-preview


 20%|██        | 2/10 [00:33<02:16, 17.02s/it]

Retry: Erreur XML pour le fichier output/niveau_parallele_2.1/gemini-3.1-flash-lite-preview/FS/TR2_p1785-1786.xml: Opening and ending tag mismatch: author line 118 and bibl, line 118, column 133 (<string>, line 118)
Retry: Erreur XML pour le fichier output/niveau_parallele_2.1/gemini-3.1-flash-lite-preview/FS/TR2_p1785-1786.xml: Opening and ending tag mismatch: author line 118 and bibl, line 118, column 133 (<string>, line 118)


 30%|███       | 3/10 [01:36<04:26, 38.08s/it]

Impossible de corriger le fichier output/niveau_parallele_2.1/gemini-3.1-flash-lite-preview/FS/TR2_p1785-1786.xml


...

Erreur openrouter: Connection error.


 90%|█████████ | 9/10 [15:32<00:17, 17.97s/it]

Erreur openrouter: Connection error.


100%|██████████| 10/10 [15:33<00:00, 93.38s/it]

Erreur openrouter: Connection error.


### B/ Programme permettant d'obtenir des fichiers xml à partir du prompt chaining, les résultats d'un niveau avec ses différent stratégies et modèles servent de base pour obtenir les sorties du niveau suivant. Par exemple, gpt-5-mini et FS du niveau 2.1 permettent d'obtenir les sorites gpt-5-mini et FS du niveau 2.2.

In [ ]:

## **** A CONFIGURER ****
provider = "openrouter" #"googleai" or "googleai"

models = ["openai/gpt-5-mini", "google/gemma-3-27b-it", "openai/gpt-5.4-mini", "google/gemini-3.1-flash-lite-preview"]
#models=["google/gemma-3-27b-it"]
#models=["openai/gpt-5-mini"]
#models=["openai/gpt-5.4-mini"]
#models=["google/gemini-3.1-flash-lite-preview"]

#Modifier le niveau des prompts à chaque fois
prompts = sorted(Path("./prompts/niveau2.8/").rglob("*.txt"))

for prompt in prompts:
    with open(prompt) as f:
        pre_prompt = f.read()
    prompt_type=prompt.stem
    print("* "+prompt_type)

    for model_name in models:
        if provider == "openrouter":
            model_short = model_name.split("/")[1].split(":")[0]
        elif provider == "googleai":
            model_short = model_name

        print("** "+ model_short)

        #Modifier selon le niveau des entrées
        ocr_path_name = f"./output/niveau2.7/{model_short}/{prompt_type}"
        ocr_files = sorted(Path(ocr_path_name).rglob("*.xml"))

        #Modifier selon le niveau des sorties
        output_path = Path(f"./output/niveau2.8/{model_short}/{prompt.stem}")
        output_path.mkdir(parents=True, exist_ok=True)

        for ocr_file in tqdm(ocr_files):
            ocr_text = ocr_file.read_text(encoding="utf-8", errors="replace")

            #Si le fichier existe déjà, passer
            output_file = output_path / f"{ocr_file.stem}.xml"
            if output_file.exists():
                continue

            corrected_text = tei_encoding(pre_prompt, ocr_text, provider, client, model_name)
            
            #Vérifier si le XML produit est valide (max 2 tentatives)
            for _ in range(2):
                try:
                    lxml.etree.fromstring(corrected_text.encode("utf-8"))
                    break
                except lxml.etree.XMLSyntaxError as e:
                    print(f"Retry: Erreur XML pour le fichier {ocr_file}: {e}")
                    corrected_text = tei_encoding(pre_prompt, ocr_text, provider, client, model_name)
            else:
                print(f"Impossible de corriger le fichier {ocr_file}")
                #continue

            #Écrire le texte valide dans le fichier de sortie
            output_file.write_text(corrected_text, encoding="utf-8", errors="replace")



* FS-R
** gpt-5-mini


100%|██████████| 10/10 [12:54<00:00, 77.41s/it]


** gemma-3-27b-it


 10%|█         | 1/10 [02:51<25:42, 171.34s/it]

Erreur openrouter: Expecting value: line 815 column 1 (char 4477)


...

100%|██████████| 10/10 [1:46:38<00:00, 639.89s/it]


** gpt-5.4-mini


100%|██████████| 10/10 [03:33<00:00, 21.36s/it]


** gemini-3.1-flash-lite-preview


100%|██████████| 10/10 [01:21<00:00,  8.17s/it]
